In [2]:
from itertools import product

# Преобразование целого числа в двоичный вектор длины m
def int_to_bits(x, m):
    return [(x >> i) & 1 for i in range(m - 1, -1, -1)]

# Построение проверочной матрицы H кода Хэмминга
# H имеет размер m x (2^m - 1)
# Столбцы — все ненулевые двоичные векторы длины m
def build_hamming_parity_check_matrix(m):
    n = 2**m - 1
    columns = [int_to_bits(i, m) for i in range(1, n + 1)]

    # преобразуем список столбцов в матрицу строк
    H = [[columns[j][i] for j in range(n)] for i in range(m)]
    return H

# Побитовое сложение по модулю 2 (XOR)
def xor_vectors(a, b):
    return [(x ^ y) for x, y in zip(a, b)]

# Вычисление линейной комбинации строк
# coeffs — коэффициенты (0 или 1)
def linear_combination(rows, coeffs):
    result = [0] * len(rows[0])
    for c, row in zip(coeffs, rows):
        if c == 1:
            result = xor_vectors(result, row)
    return result

# Генерация всех кодовых слов по порождающей матрице G
def generate_code_from_generator(G):
    k = len(G)
    codewords = []
    for u in product([0, 1], repeat=k):
        cw = linear_combination(G, u)
        codewords.append(cw)
    return codewords

# Вес Хэмминга (количество единиц)
def hamming_weight(v):
    return sum(v)

# Вычисление ранга бинарной матрицы методом Гаусса
def rank_binary_matrix(matrix):
    A = [row[:] for row in matrix]
    rows = len(A)
    cols = len(A[0])
    rank = 0
    col = 0

    while rank < rows and col < cols:
        pivot = None
        for r in range(rank, rows):
            if A[r][col] == 1:
                pivot = r
                break

        if pivot is None:
            col += 1
            continue

        A[rank], A[pivot] = A[pivot], A[rank]

        for r in range(rows):
            if r != rank and A[r][col] == 1:
                A[r] = xor_vectors(A[r], A[rank])

        rank += 1
        col += 1

    return rank

# Проверка того, что дуальный код Хэмминга является симплекс-кодом
def verify_dual_of_hamming_is_simplex(m):

    # Строим проверочную матрицу кода Хэмминга
    H = build_hamming_parity_check_matrix(m)

    # Дуальный код — это линейная оболочка строк H
    dual_code = generate_code_from_generator(H)

    n = len(H[0])
    k_dual = rank_binary_matrix(H)

    expected_n = 2**m - 1
    expected_k = m
    expected_d = 2**(m - 1)

    # исключаем нулевое слово
    nonzero_codewords = [cw for cw in dual_code if any(cw)]

    weights = [hamming_weight(cw) for cw in nonzero_codewords]
    unique_weights = sorted(set(weights))
    min_distance = min(weights)

    print("m =", m)
    print("Длина кода n =", n)
    print("Размерность дуального кода =", k_dual)
    print("Количество кодовых слов =", len(dual_code))
    print("Возможные веса ненулевых слов:", unique_weights)
    print("Минимальное расстояние =", min_distance)
    print()

    if (n == expected_n and
        k_dual == expected_k and
        unique_weights == [expected_d] and
        min_distance == expected_d):

        print("Вывод: дуальный код кода Хэмминга действительно является симплекс-кодом.")
        print("Параметры симплекс-кода: [{}, {}, {}]".format(expected_n, expected_k, expected_d))
    else:
        print("Проверка не выполнена.")

if __name__ == "__main__":
    # Проверяем для нескольких значений m
    for m in [3, 4, 5]:
        verify_dual_of_hamming_is_simplex(m)
        print("-" * 50)


m = 3
Длина кода n = 7
Размерность дуального кода = 3
Количество кодовых слов = 8
Возможные веса ненулевых слов: [4]
Минимальное расстояние = 4

Вывод: дуальный код кода Хэмминга действительно является симплекс-кодом.
Параметры симплекс-кода: [7, 3, 4]
--------------------------------------------------
m = 4
Длина кода n = 15
Размерность дуального кода = 4
Количество кодовых слов = 16
Возможные веса ненулевых слов: [8]
Минимальное расстояние = 8

Вывод: дуальный код кода Хэмминга действительно является симплекс-кодом.
Параметры симплекс-кода: [15, 4, 8]
--------------------------------------------------
m = 5
Длина кода n = 31
Размерность дуального кода = 5
Количество кодовых слов = 32
Возможные веса ненулевых слов: [16]
Минимальное расстояние = 16

Вывод: дуальный код кода Хэмминга действительно является симплекс-кодом.
Параметры симплекс-кода: [31, 5, 16]
--------------------------------------------------
